# WSP arm: Claude edits the trained probe's weights directly

θ_new = θ_old + Δθ_Claude (no retraining). Compared against gradient fine-tuning, refit, random edits and the unedited probe. Calibration and test are separate; test is unsealed once.

1. Runtime → GPU (T4). 2. Secret `ANTHROPIC_API_KEY`. 3. Run all.

In [ ]:
BRANCH = 'research/wsp-weight-edit'
!rm -rf LLM_lie_detector && git clone -q -b $BRANCH https://github.com/msverma101/LLM_lie_detector.git
%cd LLM_lie_detector/experiments/research_loop
!git log --oneline -1
!pip install -q -U transformers accelerate anthropic huggingface_hub

## 1. Activations (same cache as the research loop, ~13-20 min on a T4)

In [ ]:
!python cache_activations.py --model Qwen/Qwen3.5-2B --n_per_family 200 --out acts

## 2. Train the probe once, save θ0, freeze partitions

In [ ]:
%cd ../weight_space_edit
!python wsp.py prepare --acts ../research_loop/acts --run runs_wsp/run1

## 3. Ordinary optimization of the same parameters (GD fine-tune + refit)

In [ ]:
!python wsp.py gd --acts ../research_loop/acts --run runs_wsp/run1

## 4. WSP: Claude Opus 5.5 edits the weights (only dev_inspect + aggregate selection scores)

In [ ]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!python wsp.py claude --acts ../research_loop/acts --run runs_wsp/run1

## 5. Random-edit control, then the one-time final evaluation

In [ ]:
!python wsp.py random --acts ../research_loop/acts --run runs_wsp/run1
!python wsp.py final --acts ../research_loop/acts --run runs_wsp/run1

In [ ]:
!cd runs_wsp && zip -qr ../wsp_run1.zip run1 -x 'run1/candidates/*.npz' 'run1/*.npz'
from google.colab import files
files.download('wsp_run1.zip')